# **Instalação**

In [ ]:
!pip install -q git+https://github.com/bdcdo/raspe.git

# **Raspagem de dados**

In [ ]:
import logging
logging.getLogger("FOLHA").setLevel(logging.INFO)

import raspe

folha = raspe.folha()

dados = folha.raspar(
    pesquisa=["bolsonaro", "lula", "eleições", "eleitoral"],
    site="todos",
    data_inicio="2023-01-01",
    data_fim="2023-01-08",
)

print(f"Total bruto: {len(dados)}")

# remover duplicatas (notícias que mencionam os dois nomes)
termo_agrupado = dados.groupby("link")["termo_busca"].apply(lambda x: ", ".join(sorted(set(x))))
dados = dados.drop_duplicates(subset="link").reset_index(drop=True)
dados = dados.merge(termo_agrupado.rename("termos_busca"), on="link")
dados = dados.drop(columns=["termo_busca"])

print(f"Total após remover duplicatas: {len(dados)}")
dados.head()

2026-09-25 00:47:40,081 - FOLHA - INFO - Iniciando raspagem com parâmetros {'pesquisa': ['bolsonaro', 'lula', 'eleições', 'eleitoral'], 'site': 'todos', 'data_inicio': '2023-01-01', 'data_fim': '2023-01-08'}
2026-09-25 00:47:40,081 - FOLHA - INFO - Iniciando raspagem com parâmetros {'pesquisa': ['bolsonaro', 'lula', 'eleições', 'eleitoral'], 'site': 'todos', 'data_inicio': '2023-01-01', 'data_fim': '2023-01-08'}
2026-09-25 00:47:40,081 - FOLHA - INFO - Iniciando raspagem com parâmetros {'pesquisa': ['bolsonaro', 'lula', 'eleições', 'eleitoral'], 'site': 'todos', 'data_inicio': '2023-01-01', 'data_fim': '2023-01-08'}
2026-09-25 00:47:40,081 - FOLHA - INFO - Iniciando raspagem com parâmetros {'pesquisa': ['bolsonaro', 'lula', 'eleições', 'eleitoral'], 'site': 'todos', 'data_inicio': '2023-01-01', 'data_fim': '2023-01-08'}
2026-09-25 00:47:40,081 - FOLHA - INFO - Iniciando raspagem com parâmetros {'pesquisa': ['bolsonaro', 'lula', 'eleições', 'eleitoral'], 'site': 'todos', 'data_inicio': 

Total bruto: 1353
Total após remover duplicatas: 757


,link,titulo,resumo,data,termos_busca
0,https://www1.folha.uol.com.br/ilustrissima/202...,Ex-alunos acusam escola de arte de praticar vi...,"Em 2022, havia nas paredes do seu espaço retra...",4.jan.2023 às 8h00,bolsonaro
1,https://www1.folha.uol.com.br/mercado/2023/01/...,"Como a Apple perdeu US$ 1 trilhão em um ano, r...","Os indicadores pioraram bem à tarde, após o mi...",4.jan.2023 às 7h00,"bolsonaro, lula"
2,https://www1.folha.uol.com.br/podcasts/2023/01...,Podcast: Como começa e o que esperar da relaçã...,Depois de quatro anos de relação conflituosa e...,4.jan.2023 às 5h00,"bolsonaro, lula"
3,https://www1.folha.uol.com.br/poder/2023/01/ag...,AGU deixa defesa de processos de Bolsonaro e d...,Além de fazer a defesa de Bolsonaro nesses cas...,4.jan.2023 às 4h02,"bolsonaro, eleitoral, lula"
4,https://www1.folha.uol.com.br/colunas/eliogasp...,Lembrem-se do garçom Catalão,"Dois anos depois, elas colocaram Jair Bolsonar...",3.jan.2023 às 23h15,"bolsonaro, lula"


# **Filtrar apenas Opinião**

In [ ]:
dados_opiniao = dados[dados["link"].str.contains("/opiniao/", na=False)].reset_index(drop=True)
print(f"Notícias de Opinião: {len(dados_opiniao)}")
dados_opiniao.head()

Notícias de Opinião: 19


,link,titulo,resumo,data,termos_busca
0,https://www1.folha.uol.com.br/opiniao/2023/01/...,Bondade cara,"Existe saída correta, ainda que não indolor, p...",3.jan.2023 às 21h30,"bolsonaro, lula"
1,https://www1.folha.uol.com.br/opiniao/2023/01/...,Punhado de idiotas,"O governador Ibaneis Rocha (MDB), um bolsonari...",8.jan.2023 às 20h50,"bolsonaro, lula"
2,https://www1.folha.uol.com.br/opiniao/2023/01/...,Realismo lulista,A preocupação —realista e enraizada na experiê...,6.jan.2023 às 21h30,"bolsonaro, lula"
3,https://www1.folha.uol.com.br/opiniao/2023/01/...,No século 21,"Sob Bolsonaro, avanços civilizatórios foram tr...",7.jan.2023 às 21h30,"bolsonaro, lula"
4,https://www1.folha.uol.com.br/opiniao/2023/01/...,Novo Itamaraty,"Não cabe um isolamento à Bolsonaro, mas é prec...",7.jan.2023 às 21h30,"bolsonaro, lula"


# **Diagnóstico do HTML antes de rodar tudo**

In [ ]:
import requests
from bs4 import BeautifulSoup
import json

url_teste = dados_opiniao["link"].iloc[0]
print("URL testada:", url_teste)

headers = {"User-Agent": "Mozilla/5.0"}
r = requests.get(url_teste, headers=headers, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

print("\n--- Meta tags relevantes ---")
for m in soup.find_all("meta"):
    nome = m.get("name") or m.get("property")
    if nome and ("author" in nome.lower() or "byline" in nome.lower() or "writer" in nome.lower()):
        print(nome, "->", m.get("content"))

print("\n--- JSON-LD encontrado ---")
for script in soup.find_all("script", type="application/ld+json"):
    try:
        data = json.loads(script.string)
        print(json.dumps(data, indent=2, ensure_ascii=False)[:1000])
        print("---")
    except (json.JSONDecodeError, TypeError):
        continue

print("\n--- Elementos com 'autor'/'author' na classe ---")
for tag in soup.find_all(class_=True):
    classes = " ".join(tag.get("class", []))
    if "autor" in classes.lower() or "author" in classes.lower():
        print(tag.name, classes, "->", tag.get_text(strip=True)[:100])

print("\n--- Links para /colunistas/ ou /colunas/ ---")
for a in soup.find_all("a", href=True):
    if "/colunistas/" in a["href"] or "/colunas/" in a["href"]:
        print(a["href"], "->", a.get_text(strip=True))

URL testada: https://www1.folha.uol.com.br/opiniao/2023/01/bondade-cara.shtml

--- Meta tags relevantes ---

--- JSON-LD encontrado ---
{
  "@context": "http://schema.org",
  "@type": [
    "CreativeWork",
    "ReportageNewsArticle"
  ],
  "url": "https://www1.folha.uol.com.br/opiniao/2023/01/bondade-cara.shtml",
  "mainEntityOfPage": "https://www1.folha.uol.com.br/opiniao/2023/01/bondade-cara.shtml",
  "headline": "Bondade cara",
  "description": "Ao prorrogar desoneração de combustíveis, Lula erra e eleva dúvidas sobre Haddad",
  "datePublished": "2023-01-03T21:30:00Z",
  "image": {
    "@type": "ImageObject",
    "url": "https://f.i.uol.com.br/fotografia/2023/01/03/167278745063b4b5fad3532_1672787450_3x2_md.jpg",
    "width": "768",
    "height": "512"
  },
  "author": {
    "@type": "NewsMediaOrganization",
    "@id": "https://www1.folha.uol.com.br#organization",
    "name": "folha.uol.com.br",
    "url": "https://www.folha.uol.com.br/",
    "logo": {
      "@type": "ImageObject",
 

# **Coleta de autor e texto**




In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time

GENERICOS = {"folha.uol.com.br", "folha de s.paulo", "folha de sao paulo", "uol", "folhapress"}

def pegar_autor_e_texto(url, timeout=15):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        r = requests.get(url, headers=headers, timeout=timeout)
        r.encoding = "utf-8"  # corrige o problema de acentuação
        soup = BeautifulSoup(r.text, "html.parser")

        # --- autor ---
        autor = None
        candidatos = []
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.string)
                if isinstance(data, dict) and "author" in data:
                    a = data["author"]
                    if isinstance(a, dict) and a.get("name"):
                        candidatos.append(a["name"].strip())
                    elif isinstance(a, list):
                        for item in a:
                            if isinstance(item, dict) and item.get("name"):
                                candidatos.append(item["name"].strip())
            except (json.JSONDecodeError, TypeError):
                continue

        meta = soup.find("meta", attrs={"name": "author"})
        if meta and meta.get("content"):
            candidatos.append(meta["content"].strip())

        for c in candidatos:
            if c.lower() not in GENERICOS:
                autor = c
                break

        # --- texto ---
        texto = None
        corpo = soup.find("div", class_="c-news__body")
        if corpo:
            texto = corpo.get_text(" ", strip=True)

        return autor, texto

    except requests.RequestException:
        return None, None


autores = []
textos = []
total = len(dados_opiniao)

for i, link in enumerate(dados_opiniao["link"], start=1):
    autor, texto = pegar_autor_e_texto(link)
    autores.append(autor)
    textos.append(texto)
    if i % 10 == 0 or i == total:
        print(f"{i}/{total} processadas")
    time.sleep(1)

dados_opiniao["autor"] = autores
dados_opiniao["texto"] = textos
dados_opiniao.head()

10/19 processadas
19/19 processadas


,link,titulo,resumo,data,termos_busca,autor,texto
0,https://www1.folha.uol.com.br/opiniao/2023/01/...,Bondade cara,"Existe saída correta, ainda que não indolor, p...",3.jan.2023 às 21h30,"bolsonaro, lula",None,"Existe saída correta, ainda que não indolor, p..."
1,https://www1.folha.uol.com.br/opiniao/2023/01/...,Punhado de idiotas,"O governador Ibaneis Rocha (MDB), um bolsonari...",8.jan.2023 às 20h50,"bolsonaro, lula",None,O punhado de imbecis criminosos que vandalizou...
2,https://www1.folha.uol.com.br/opiniao/2023/01/...,Realismo lulista,A preocupação —realista e enraizada na experiê...,6.jan.2023 às 21h30,"bolsonaro, lula",None,"Na primeira reunião ministerial após a posse, ..."
3,https://www1.folha.uol.com.br/opiniao/2023/01/...,No século 21,"Sob Bolsonaro, avanços civilizatórios foram tr...",7.jan.2023 às 21h30,"bolsonaro, lula",None,De todos os retrocessos obscurantistas patroci...
4,https://www1.folha.uol.com.br/opiniao/2023/01/...,Novo Itamaraty,"Não cabe um isolamento à Bolsonaro, mas é prec...",7.jan.2023 às 21h30,"bolsonaro, lula",None,Um momento definidor da política externa brasi...


# **Remover linhas sem autor (editoriais) e checar texto**

In [ ]:
antes = len(dados_opiniao)
dados_opiniao = dados_opiniao[dados_opiniao["autor"].notna()].reset_index(drop=True)
depois = len(dados_opiniao)
print(f"Removidos {antes - depois} editoriais sem autor. Restaram {depois} notícias assinadas.")
dados_opiniao.head()

Removidos 12 editoriais sem autor. Restaram 7 notícias assinadas.


,link,titulo,resumo,data,termos_busca,autor,texto
0,https://www1.folha.uol.com.br/opiniao/2023/01/...,Indulto no massacre do Carandiru é despropósito,O incentivo à violência policial foi uma das m...,4.jan.2023 às 21h00,bolsonaro,Ademar Borges,Indultar é extinguir a pena aplicada a alguém ...
1,https://www1.folha.uol.com.br/opiniao/2023/01/...,Perspectivas e propostas para o sindicalismo,As reformas trabalhistas dos governos Michel T...,2.jan.2023 às 21h00,"bolsonaro, lula",Miguel Torres,O ano de 2023 inicia-se com novo governo e nov...
2,https://www1.folha.uol.com.br/opiniao/2023/01/...,Crimes de Jair Bolsonaro não podem ficar impunes,"Ainda que tenha sido eleito democraticamente, ...",1º.jan.2023 às 21h00,bolsonaro,Rogério Sottili,"De 1964 a 1985, uma série de graves crimes con..."
3,https://www1.folha.uol.com.br/opiniao/2023/01/...,Desafios para o novo MEC,Os resultados do Censo da Educação Superior 20...,2.jan.2023 às 21h00,lula,Lúcia Teixeira,Os resultados do Censo da Educação Superior 20...
4,https://www1.folha.uol.com.br/opiniao/2023/01/...,Ministério dos Povos Indígenas é reparação his...,A criação sem precedentes de um Ministério dos...,3.jan.2023 às 21h00,lula,Shirley Krenak,A criação sem precedentes de um Ministério dos...


# **Salvar**

In [ ]:
dados_opiniao.to_excel("folha_opiniao.xlsx", index=False)
from google.colab import files
files.download("folha_opiniao.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>